# Agent | Tools

**1. 도구 (Tool)**

https://docs.langchain.com/oss/python/integrations/tools/index#tools-and-toolkits

> "LLM이 사용할 수 있는 구체적인 기술이나 장비"

LLM은 기본적으로 학습된 데이터 내에서만 답변할 수 있으며, 실시간 정보나 정확한 수학 계산에는 취약하다. **Tool**은 이러한 한계를 보완하기 위해 LLM에게 쥐여주는 외부 기능이다.

* **역활:** 외부 API 호출, 웹 검색, 코드 실행, 파일 시스템 접근 등 LLM이 직접 할 수 없는 작업을 대신 수행한다.
* **예시:**
* `Google Search`: 최신 정보를 검색한다.
* `Calculator`: 정확한 수학 계산을 수행한다.
* `Python REPL`: 파이썬 코드를 작성하고 실행한다.



**2. 에이전트 (Agent)**

> "도구를 언제, 어떻게 사용할지 결정하는 두뇌"

**Agent**는 LLM을 추론 엔진(Reasoning Engine)으로 사용하여 사용자의 요청을 해결하기 위한 계획을 세우고 실행하는 주체이다. 단순히 정해진 코드를 순서대로 실행하는 것이 아니라, 상황에 따라 유연하게 행동을 결정한다.

* **역활:** 사용자의 질문을 분석하고, 어떤 **Tool**이 필요한지 판단(Thought)하고, 해당 도구를 실행(Action)한 뒤, 그 결과(Observation)를 보고 다음 행동을 결정하거나 최종 답변을 내놓는다.
* **작동 방식 (ReAct 패턴 예시):**
1. **질문:** "현재 서울 날씨에 맞는 옷차림 추천해줘."
2. **생각(Thought):** "서울의 현재 날씨를 먼저 알아야 한다." -> `Search` 도구 선택
3. **행동(Action):** `Search("서울 현재 날씨")` 실행
4. **관찰(Observation):** "서울 기온 5도, 맑음"이라는 결과 획득
5. **생각(Thought):** "5도면 코트나 패딩이 필요하다."
6. **최종 답변:** "현재 서울은 5도이므로 코트나 가벼운 패딩을 추천합니다."

**Agent와 Tools의 상호작용**

1. **Agent가 입력을 받음**: 사용자의 요청을 LLM으로 분석.
2. **적합한 Tool 선택**: 요청을 처리하는 데 가장 적합한 Tool을 선택.
3. **Tool 실행 및 결과 반환**: Tool을 실행하고 결과를 받아 사용자에게 응답.

In [ ]:
# %pip install -Uqqq langchain_openai langchain_community langchain_tavily langgraph wikipedia numexpr arxiv ddgs

Note: you may need to restart the kernel to use updated packages.


In [31]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

## Tools

In [32]:
import importlib, pkgutil # 모듈 동적 로드 / 패키지 탐색 유틸

# langchain_community.tools 패키지 로드
package = importlib.import_module('langchain_community.tools')

# 해당 패키지 경로 아래의 하위 모듈들을 하나씩 순회
for module in pkgutil.iter_modules(package.__path__):
    print(module.name) # 각 모듈(도구) 이름 출력

ainetwork
amadeus
arxiv
asknews
audio
azure_ai_services
azure_cognitive_services
bearly
bing_search
brave_search
cassandra_database
clickup
cogniswitch
connery
convert_to_openai
databricks
dataforseo_api_search
dataherald
ddg_search
e2b_data_analysis
edenai
eleven_labs
few_shot
file_management
financial_datasets
github
gitlab
gmail
golden_query
google_books
google_cloud
google_finance
google_jobs
google_lens
google_scholar
google_serper
google_trends
graphql
human
ifttt
interaction
jina_search
jira
json
memorize
merriam_webster
metaphor_search
mojeek_search
multion
nasa
nuclia
office365
openai_dalle_image_generation
openapi
openweathermap
passio_nutrition_ai
playwright
plugin
polygon
powerbi
pubmed
render
requests
riza
scenexplain
searchapi
searx_search
semanticscholar
shell
slack
sleep
spark_sql
sql_database
stackexchange
steam
steamship_image_generation
tavily_search
vectorstore
wikidata
wikipedia
wolfram_alpha
yahoo_finance_news
you
youtube
zapier
zenguard


### wikipedia Tool

In [44]:
from datetime import timedelta
import wikipedia

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# 반드시 본인의 실제 연락 가능한 주소로 변경
wikipedia.set_user_agent(
    "playdata-langchain/1.0 (https://github.com/ilil1)"
)

# 연속 요청 제한
wikipedia.set_rate_limiting(
    True,
    min_wait=timedelta(seconds=2),
)

wiki_api = WikipediaAPIWrapper(
    lang="ko",
    top_k_results=1,
    doc_content_chars_max=2000,
)

wiki_tool = WikipediaQueryRun(api_wrapper=wiki_api)

print(wiki_tool.invoke("피지컬 AI"))

Page: 피지컬 AI
Summary: 피지컬AI(physical AI) 또는 생성형 피지컬 AI는 로봇, 자율주행차, 스마트 공간 등 자율 시스템이 실제 물리 세계에서 사물을 인지하고, 이해하며, 복잡한 행동을 수행할 수 있도록 해주는 기술이다. 피지컬 AI는 3D 세계의 공간적 관계와 물리적 작동 방식에 이해를 기반으로 현재의 생성형 AI를 확장한다.
주요 기술 수준과 형태에 따라 휴머노이드형, 자율주행차형, 드론형, AGV&AMR형으로 분류되어 다양한 산업 환경에 특화된 형태로 활용된다.
피지컬 AI를 다룬 대표적인 책은 피지컬 AI 메가 트렌드, 피지컬AI 패권전쟁, AI 다음 물결등이 있다.




In [ ]:
# from langchain_community.tools import WikipediaQueryRun         # 위키피디아 질문 실행 Tool
# # 위키피디아 검색/요약 API 요청 래퍼 클래스
# from langchain_community.utilities import WikipediaAPIWrapper   

# # 위키피디아 API 래퍼를 Tool에 연결
# wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
# print(wiki_tool.run('피지컬 AI')) # 위키피디아 검색/요약 결과 출력

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [42]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from pprint import pprint

messages = [('human', '걸그룹 튜이드 멤버 알려줘')]
llm = init_chat_model('gpt-5.4-mini')
print(llm.invoke('걸그룹 튜이드 멤버 알려줘')) # 최신정보 알지 못함

agent = create_agent(
    model = llm,
    tools = [wiki_tool]
)

response = agent.invoke({'messages': messages})

pprint(response)

content='혹시 **걸그룹 “트와이스(TWICE)”**를 말씀하신 걸까요?  \n“튜이드”라는 이름의 걸그룹은 제가 바로 확인하기가 어려워요.\n\n만약 **트와이스 멤버**를 원하신 거라면 현재 멤버는:\n\n- 나연\n- 정연\n- 모모\n- 사나\n- 지효\n- 미나\n- 다현\n- 채영\n- 쯔위\n\n원하시면  \n1) **멤버 소개**  \n2) **각 멤버 본명/국적**  \n3) **최근 활동**  \n중 하나로 이어서 알려드릴게요.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 17, 'total_tokens': 169, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHLEFV8QwinPtB60wnaeBfbdU3eXz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0414b-dcc0-7520-afad-c5e8519dbd98-0' tool_calls=[] invalid_tool_calls=[] usa

In [ ]:
print(response['messages'][-1].content)

혹시 **걸그룹 “튜이드”**를 말씀하신 걸까요?  
제가 확인한 바로는 **“튜이드(Tweed)”라는 이름의 유명 걸그룹 정보가 바로 나오지 않아서**, 정확한 팀명을 다시 알려주시면 멤버를 찾아드릴게요.

예를 들면:
- **트와이스(TWICE)**
- **뉴진스(NewJeans)**
- **르세라핌(LE SSERAFIM)**
- **아이브(IVE)**

원하시면 제가 **“튜이드”가 맞는지 철자 포함해서** 다시 확인해드릴게요.


In [ ]:
llm = init_chat_model('gpt-5.4-mini')
print(llm.invoke('한국 그룹 롱샷 멤버 알려줘')) # 최신정보 알지 못함


content='한국 그룹 **롱샷(Long Shot)**의 멤버는 보통 다음으로 알려져 있습니다:\n\n- **유진**\n- **태성**\n- **민우**\n- **지훈**\n\n다만 **동명이인/동명 그룹**이 있을 수 있고, 활동 시기나 자료에 따라 멤버 구성이 다르게 나올 수 있어요.  \n원하시면 제가 **해당 그룹의 결성 시기, 대표곡, 멤버별 활동**까지 같이 정리해드릴게요.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 116, 'prompt_tokens': 17, 'total_tokens': 133, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHKtntdsOk1S62CBBdIwBWyJ9M3mw', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a04138-81c1-7be3-9357-7f6b41e576e3-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 17, 'out

### load_tools

**load_tools 사용가능 목록**

라이브러리를 통해 제공되는 외부 tool들을 langchain-community에서 통합하여 사용할 수 있다.
웬만한 기능들은 langchain 생태계 내에서 쉽게 쓸 수 있다.

https://docs.langchain.com/oss/python/integrations/providers/overview

https://docs.langchain.com/oss/python/integrations/tools

| 도구 이름        | 기능 예시               |
|------------------|------------------------|
| llm-math         | LLM 기반 수학 계산     |
| wikipedia        | 위키백과 검색          |
| serpapi          | 구글 검색 API          |
| requests_get     | HTTP GET 요청          |
| requests_post    | HTTP POST 요청         |
| arxiv            | arXiv 논문 검색        |
| pubmed           | PubMed 논문 검색       |
| dalle            | DALL-E 이미지 생성     |
| bing_search      | Bing 검색              |
| duckduckgo_search| DuckDuckGo 검색        |

### arxiv

In [45]:
from langchain_community.agent_toolkits.load_tools import load_tools

llm = init_chat_model('gpt-5.4-mini')
tools = load_tools(['arxiv', 'wikipedia'])

agent = create_agent(
    model = llm,
    tools = [wiki_tool],
    system_prompt="당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해주세요.",

)

messages = [('human', '(9711003) 이 논문의 내용을 간단하게 설명해줄래? (한글 답변)')]
response = agent.invoke({'messages': messages})

pprint(response)
print("="*50)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='(9711003) 이 논문의 내용을 간단하게 설명해줄래? (한글 답변)', additional_kwargs={}, response_metadata={}, id='ce5f61c9-d5eb-481b-9f3c-9af9495ce68b'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 224, 'total_tokens': 243, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHLJ648haSRZcX6XUny2hqHevJX0L', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04150-7510-7933-a5fb-f5f2018f0694-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': '9711003 paper'}, 'id': 'call_8FE

In [48]:
import requests # HTTP 요청 보내는 라이브러리
import xml.etree.ElementTree as ET
from langchain_core.tools import tool

@tool
def search_arxiv(arxiv_id: str) -> str:
    """arXiv 논문 ID 로 제목, 저자, 초록을 조회합니다."""

    url = "https://export.arxiv.org/api/query"
    response = requests.get(
        url, 
        params = {
            "id_list" : arxiv_id, # 논문 ID
            "max_result" : 1 # 결과 1개
        },
        timeout = 10 # 응답 대기시간 
    )

    response.raise_for_status() # 요청 실패시 예외 발생

    root = ET.fromstring(response.text) # xml 문자열을 받아 Element 객체로 변환

    ns = {"atom": "http://www.w3.org/2005/Atom"} # arXiv 응답의 XML 네임스페이스
    entry = root.find("atom:entry", ns) # 논문 정보가 담긴 entry 태그

    if entry is None:
        return "논문 정보를 찾을 수 없습니다."

    title = entry.findtext("atom:title", namespaces=ns).strip() # 논문 제목 추출
    summary = entry.findtext("atom:summary", namespaces=ns).strip() # 요약 정보 추출
    # 저자들 추출
    authors = [author.findtext("atom:name", namespaces =ns)
               for author in entry.findall("atom:author", ns)] 


    return f"""
제목 : {title}
저자 : {', '.join(authors)}
초록 : {summary} 
"""

In [49]:
tools = [search_arxiv, wiki_tool]

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt="당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요."
)

messages = [('human','1706.03762 이 논문의 내용을 간단하게 설명해줄래? (한글답변)')]
response = agent.invoke({'messages' : messages})
print(response['messages'][-1].content)



네. **arXiv:1706.03762**는 유명한 논문 **“Attention Is All You Need”**로, **Transformer**라는 새로운 신경망 구조를 제안한 논문입니다.

### 아주 간단히 말하면
기존의 번역 모델들은 주로 **RNN**이나 **CNN**을 사용했는데, 이 논문은 그런 구조 없이 **Attention 메커니즘만으로**도 훨씬 잘 동작하는 모델을 만들 수 있다고 보여줬습니다.

### 핵심 내용
- **Transformer**라는 모델을 제안함
- **순환 구조(RNN)**나 **합성곱(CNN)** 없이, **Attention**만 사용
- 문장 내에서 어떤 단어가 중요한지 직접 참고하면서 처리해서  
  **병렬 처리에 유리하고 학습 속도가 빠름**
- 기계번역에서 기존 최고 성능보다 더 좋은 결과를 냄
- 영어-독일어, 영어-프랑스어 번역에서 좋은 성능을 기록함

### 왜 중요한가?
이 논문은 오늘날 **BERT, GPT 같은 대형 언어 모델들의 기반이 되는 핵심 아이디어**를 제시한 매우 중요한 논문입니다.

원하시면 제가 이 논문을  
1) **그림 없이 직관적으로 설명**하거나,  
2) **Attention이 무엇인지부터** 쉽게 풀어서 설명해드릴게요.


### llm-math

In [ ]:
from langchain_community.agent_toolkits.load_tools import load_tools

llm = init_chat_model('gpt-4.1-mini')

# wikipedia, llm-math 도구 로드 (수학 도구는 계산용 llm 필요)
tools = load_tools(['wikipedia', 'llm-math'], llm =llm)

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt="""당신은 현명한 챗봇입니다. 
    주어진 도구를 적절하게 활용하여, 답변해 주세요.
    단, 숫자계산은 반드시 llm-math 도구를 사용해서 답변에 활용해야 합니다.
    """
)

response = agent.invoke({'messages' : '3.5의 3제곱은 몇이야? 그리고 그 결과에 5를 곱해줘.'})
pprint(response)
print("=" * 50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='3.5의 3제곱은 몇이야? 그리고 그 결과에 5를 곱해줘.', additional_kwargs={}, response_metadata={}, id='c615080b-0302-4420-bdce-f4f03e107f07'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 194, 'total_tokens': 253, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5cfb5b86aa', 'id': 'chatcmpl-EHNadvZVdvCHKXCPMJyB1YsQ1MxWU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a041d6-5c2a-7db3-b0ca-a91af23fe1c0-0', tool_calls=[{'name': 'Calculator', 'args': {'__arg1': '3.5^3'}, 'id': 'call_Z9dQM

### duckduckgo
https://reference.langchain.com/python/langchain-community/tools/ddg_search/tool/DuckDuckGoSearchRun

DuckDuckGo는 개인정보 추적 없이 웹 검색을 제공하는 검색 엔진으로,
LangChain에서는 이를 외부 최신 정보 검색용 Tool로 활용한다.

실시간 웹 검색 가능
→ LLM의 knowledge cutoff 이후 이슈(뉴스, 화제, 트렌드)에 대응 가능
로그인/API 키 불필요
→ 실습·교육 환경에서 바로 사용 가능
프라이버시 중심
→ 사용자 검색 이력 추적 없음

LangChain에서 제공하는 DuckDuckGo Tool 차이
- DuckDuckGoSearchRun
    - 검색 결과를 하나의 텍스트 요약으로 반환
    - 빠른 질의응답용에 적합
- DuckDuckGoSearchResults
    - 검색 결과를 리스트(제목, 링크, 스니펫 등 구조화) 형태로 반환
    - 에이전트가 여러 결과를 비교·판단해야 할 때 유리

In [ ]:
# 덕덕고 검색 Tool 2종류
from langchain_community.tools import DuckDuckGoSearchRun, DuckDuckGoSearchResults

ddgs = DuckDuckGoSearchRun() # 검색 결과를 텍스트 요약 형태로 반환
print(ddgs.invoke("Trump's first name?")) # 문자열 출력

ddgs2 = DuckDuckGoSearchResults() # 검색 결과를 구조화 된 리스트로 반환
# 제목/링크/스니펫 정보등의 리스트로 반환
print(ddgs.invoke("Trump's first name?"))


The company name "E. Trump & Son" appeared in advertising by 1924, [44] by which year Trump ostensibly used an $800 loan from his mother to complete and sell his first house. [45][37][46] Public records, however, do not support him building until 1927, [47] the year the company was incorporated [48] (and following Trump's 21st birthday). The Trump family is a prominent wealthy American family. The best-known member is patriarch Donald Trump, the 45th and current 47th president of the United States (2017-2021, 2025-present). The Trumps are of German descent. [1] They are active in business, entertainment, politics, and real estate. Other prominent members include Donald Trump's father Fred Trump, and grandfather Frederick ... Trump was sworn in as president on January 20, 2017. During his first term, his administration focused on immigration, trade, tax cuts, and reducing government regulations. Trump withdrew the United States from the Trans-Pacific Partnership and announced that the c

In [ ]:
llm = init_chat_model('gpt-5.4-mini')

# wikipedia, llm-math 도구 로드 (수학 도구는 계산용 llm 필요)
tools = [ddgs2]
# tools = load_tools([ddgs2], llm =llm)

agent = create_agent(
    llm, tools, 
    system_prompt='모르는 정보가 있으면 ddgs tool을 사용해 검색해.')

response = agent.invoke({'messages' : 'gs25 민음사 빵'})
pprint(response)
print("=" * 50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='gs25 민음사 빵', additional_kwargs={}, response_metadata={}, id='b3d66e19-ee4c-4c8d-9f2f-1905b53291ec'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 180, 'total_tokens': 207, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHNs9L1XHYqMzGuh1DrLG3XcFhWNX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a041e6-ecc7-7412-95a5-c15bcad5470e-0', tool_calls=[{'name': 'duckduckgo_results_json', 'args': {'query': 'GS25 민음사 빵'}, 'id': 'call_I3ERLgFrr0jDlWSvD0Jh3

### tavily-search

https://docs.langchain.com/oss/python/integrations/tools/tavily_search

In [ ]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_results = 3,
    topic = 'general', # general/news/finance 등 선택
    include_images = True, # 이미지 URL 함께 반환
    search_depth = 'advanced' # basic/advanced (advanced 는 더 깊게 찾음)
)

tavily_tool.invoke('2026년 8월 현재 대한민국에서 가장 핫한 이슈가 뭐야?')


{'query': '2026년 8월 현재 대한민국에서 가장 핫한 이슈가 뭐야?',
 'follow_up_questions': None,
 'answer': None,
 'images': ['https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426f67d1b5fa839e454dfe_79_thumbnail2.png',
  'https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426e7fc594ca778cc36ab8_79_2-1.png',
  'https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426ed2d1b5fa839e44e068_79_2-2.png',
  'https://cdn.wakeupnews.co.kr/news/photo/202601/973_1856_5154.png',
  'https://lookaside.instagram.com/seo/google_widget/crawler/?media_id=3956261914254161255'],
 'results': [{'url': 'https://www.newsshin.co.kr/news/articleView.html?idxno=',
   'title': "【뉴스신ㅣ2026년 8월 22일(토) ㅣ대한민국 '핫' 이슈】",
   'content': '【뉴스신】2026년 8월 22일 대한민국은 기술의 속도, 경제의 온도, 정치의 긴장, 사회의 불안이 동시에 표출된 하루였다.세계는 AI가 인간의 통제를',
   'score': 0.9041187,
   'raw_content': None,
   'images': [],
   'id': '554e9c-00'},
  {'url': 'https://www.korea.kr',
   'title': '대한민국 정책브리핑',
   'content': '여행   여행을 사랑하는 국민 4만 명이 직접 뽑은 

In [ ]:
llm = init_chat_model('gpt-5.4-mini')

# wikipedia, llm-math 도구 로드 (수학 도구는 계산용 llm 필요)
tools = [tavily_tool]

agent = create_agent(
    llm, tools, 
    system_prompt='''
당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여,
사용자의 질문에 거짓없이 답변을 해주세요.
''')

response = agent.invoke({'messages' : '현재 AI업계에서 가장 핫한 주제가 뭐야?'})
pprint(response)
print("=" * 50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='현재 AI업계에서 가장 핫한 주제가 뭐야?', additional_kwargs={}, response_metadata={}, id='6eed0ebe-127f-4909-a19f-44a215ca8b10'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 1321, 'total_tokens': 1377, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHO3HHtiQdy5chwFTzm3xEX32Yns0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a041f1-72e2-7732-9531-345e7fbf2456-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': "current hottest topics in AI industry 2026

In [64]:
llm = init_chat_model('gpt-5.4-mini')

# wikipedia, llm-math 도구 로드 (수학 도구는 계산용 llm 필요)
tools = [tavily_tool]

agent = create_agent(
    llm, tools, 
    system_prompt='''
당신은 미국주식시장 분석봇입니다.
사용자가 요청한 기업에 대한 2026년 보고서를 직관적으로 분석해주세요.

# 출력형식
다음 내용을 포함해 표형식 출력 (분석기관별 레코드로 작성)

1. 분석기관명
2. 목표주가범위 (최저 ~ 최대)
3. 전망근거 키워드
4. 신뢰도 지수(1 ~ 10)
''')

response = agent.invoke({'messages' : '2026년 상승할 가능성이 가장 높은 미국주식 분석해 줘'})
pprint(response)
print("=" * 50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='2026년 상승할 가능성이 가장 높은 미국주식 분석해 줘', additional_kwargs={}, response_metadata={}, id='710ec8b0-28fb-43ab-901c-432459964397'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 1384, 'total_tokens': 1438, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHO5E5tdfR9p8K0lu6aLyL41WicBW', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a041f3-4c16-7ea1-91a8-0836a748f782-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '2026 year ahead stock market outlo

In [ ]:
from IPython.display import display, Markdown

# 마지막 메시지 내용을 Markdown 변환 후 보기 좋게 출력
display(Markdown(response['messages'][-1].content))

아래는 **2026년 상승 가능성이 높다고 판단되는 미국주식 후보**를, 최근 애널리스트 전망과 상승 논리를 바탕으로 **기관별 레코드 형식**으로 정리한 표입니다.  
(참고: 실제 투자 판단은 변동성·밸류에이션·실적 리스크를 함께 보셔야 합니다.)

| 분석기관명 | 목표주가범위 (최저 ~ 최대) | 전망근거 키워드 | 신뢰도 지수(1~10) |
|---|---:|---|---:|
| Goldman Sachs | $7,600 ~ $7,600 (S&P 500 기준) | AI 투자, EPS 성장, 밸류에이션 정상화, 2026 실적 모멘텀 | 8 |
| 24/7 Wall St. | $245.91 ~ $357 (NVIDIA, NVDA) | AI 인프라, 데이터센터 수요, GPU 리더십, 높은 애널리스트 우호도 | 8 |
| Wall Street 컨센서스(종합) | $250 ~ $352 (NVIDIA, NVDA) | 강한 매수 의견, AI CAPEX, 마진 방어, 플랫폼 지배력 | 9 |
| Barchart/FinancialContent | $330.46 ~ $500 (Micron, MU) | AI 메모리 수요, HBM, EPS 급성장, 반도체 업사이클 | 8 |
| 24/7 Wall St. | $503.61 ~ $503.61 (Broadcom, AVGO) | AI 반도체, 커스텀 실리콘, 네트워킹, 현금흐름 확대 | 8 |
| 시장 컨센서스(종합) | $412 ~ $523.73 (Broadcom, AVGO) | AI 인프라, 하이퍼스케일러 수요, 강한 Buy 비중, 실적 서프라이즈 | 8 |
| MarketBeat/애널리스트 종합 | $305 ~ $370 (Amazon, AMZN) | AWS 재가속, AI 클라우드, 광고 성장, 리테일 마진 개선 | 9 |
| TD Cowen / Barclays 등 | $310 ~ $350 (Amazon, AMZN) | AWS 성장, AI 워크로드 이전, CAPEX 확대, 플랫폼 점유율 | 8 |
| ChartMill | $971.73 ~ $971.73 (Sterling Infrastructure, STRL) | 인프라 투자, 수주 성장, 실적 모멘텀, 애널리스트 업사이드 | 7 |
| ChartMill | $125.21 ~ $125.21 (Innodata, INOD) | AI 데이터 서비스, 성장률 급등, 소형주 고베타, 모멘텀 | 6 |

### 한줄 결론
- **가장 상승 가능성이 높아 보이는 메가테마는 AI 반도체/인프라**
- 그중 **NVIDIA(NVDA), Broadcom(AVGO), Amazon(AMZN), Micron(MU)** 이 2026년에도 가장 강한 후보로 보입니다.
- **상승 여력만 보면 소형주(STRL, INOD)** 도 크지만, **신뢰도는 대형주보다 낮습니다.**

원하시면 다음 단계로  
**“2026년 상승 가능성 TOP 10 미국주식”**을 제가 **점수화해서 순위표**로 정리해드릴게요.

### @tool

In [ ]:
# eval / exec로 문자열 코드 실행

a = 10
print(eval("5+3+a")) # 문자열을 평가해서 결과를 반환
exec("b=10")         # 문자열을 실행 (할당 가능)
print(b)


18
10


In [ ]:
from langchain_core.tools import tool

@tool
def simple_calculator(query: str) -> str:
    """
    산술연산을 위한 간단한 계산기 Tool
    Args:
        query: 계산식
    Return:
        계산식 결과값

    Examples:
    - simple_calculator("5 + 3 - 2") -> "계산 결과: 6"
    - simple_calculator("4 ** 2 / 8") -> "계산 결과: 2"
    """

    try:
        result = eval(query)    # 문자열을 eval로 평가(결과 반환)
        return f"계산 결과 : {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"


simple_calculator    

StructuredTool(name='simple_calculator', description='산술연산을 위한 간단한 계산기 Tool\nArgs:\n    query: 계산식\nReturn:\n    계산식 결과값\n\nExamples:\n- simple_calculator("5 + 3 - 2") -> "계산 결과: 6"\n- simple_calculator("4 ** 2 / 8") -> "계산 결과: 2"', args_schema=<class 'langchain_core.utils.pydantic.simple_calculator'>, func=<function simple_calculator at 0x00000196A4592660>)

In [68]:
llm = init_chat_model('gpt-5.4-mini')

tools = [simple_calculator]

agent = create_agent(
    llm, tools, 
    system_prompt='''
당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여,
사용자의 질문에 거짓없이 답변을 해주세요.
''')

response = agent.invoke({'messages' : '7 + 3 * 8 이거를 계산해 줘.'})
pprint(response)
print("=" * 50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='7 + 3 * 8 이거를 계산해 줘.', additional_kwargs={}, response_metadata={}, id='e81c48d3-6bfd-4060-af77-52e1881ecf84'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 249, 'total_tokens': 273, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHOWhm0oZxwBETGkesKzB0devufq5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0420d-480a-72b2-aefe-54cb8381bd07-0', tool_calls=[{'name': 'simple_calculator', 'args': {'query': '7 + 3 * 8'}, 'id': 'call_YnQJZin1KnRdwK0Fmk

In [69]:
response = agent.invoke({'messages' : '김치볶음밥 레시피?'})
pprint(response)
print("=" * 50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='김치볶음밥 레시피?', additional_kwargs={}, response_metadata={}, id='bdb73566-7ec3-40e4-bb8f-f7fd41b9cb51'),
              AIMessage(content='물론이죠. 간단한 김치볶음밥 레시피 알려드릴게요.\n\n### 재료 (1~2인분)\n- 밥 1공기\n- 김치 1/2컵~1컵\n- 대파 조금\n- 식용유 1~2큰술\n- 고추장 1작은술 또는 고춧가루 약간(선택)\n- 간장 1작은술\n- 설탕 1/2작은술\n- 참기름 1작은술\n- 김가루, 계란후라이(선택)\n\n### 만드는 방법\n1. **김치 썰기**\n   - 김치는 잘게 잘라주세요.\n\n2. **파 볶기**\n   - 팬에 기름을 두르고 대파를 먼저 볶아 향을 냅니다.\n\n3. **김치 볶기**\n   - 김치를 넣고 1~2분 볶아주세요.\n   - 신김치면 더 맛있습니다.\n\n4. **양념 넣기**\n   - 간장, 설탕을 넣고 섞어 볶습니다.\n   - 원하면 고추장이나 고춧가루를 조금 넣어도 좋아요.\n\n5. **밥 넣기**\n   - 밥을 넣고 주걱으로 잘 풀어가며 볶아줍니다.\n   - 밥이 고르게 섞이도록 2~3분 볶아주세요.\n\n6. **마무리**\n   - 마지막에 참기름을 넣고 한 번 섞습니다.\n   - 그릇에 담고 김가루나 계란후라이를 올리면 완성!\n\n### 팁\n- 밥은 **찬밥**이 더 볶기 좋습니다.\n- 김치가 너무 시면 **설탕을 조금** 넣으면 맛이 부드러워져요.\n- 햄, 참치, 베이컨을 넣으면 더 맛있습니다.\n\n원하시면 **햄 넣는 버전**, **참치 넣는 버전**, **초간단 5분 레시피**로도 알려드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 470, 'prompt_to

In [71]:
OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')

In [ ]:
import json

def get_current_weather(city="Seoul", units="metric"):
    """
    OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수

    Args:
        - city: str 날씨정보를 가져올 도시 이름. **반드시 영문으로 작성하세요.**
            - 변환예시:
                - 서울 -> Seoul
                - 충남, 충청남도 -> Chungcheongnam-do
                - 부산 -> Busan
        - units: str 온도단위를 설정하는 문자열
          - metric(기본값: 섭씨, 미터)
          - imperial(화씨, 야드)
    Return:
        - str: json 형식으로 변환된 현재 날씨 정보
    """

    url = f'https://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHER_API_KEY}&units={units}'
    response = requests.get(url)
    data = response.json() # json -> dict
    weather_info = {}

    if response.status_code == 200: # 정상 응답 받은 경우
        weather_description = data['weather'][0]['description'] # 날씨 설명
        temp = data['main']['temp'] # 현재 기온
        temp_feels_like = data['main']['feels_like'] # 체감 온도
        humidity = data['main']['humidity'] # 습도

        weather_info = {
            'city' : city,
            'description': weather_description,
            'temperature': temp,
            'temperature_feels_like' : temp_feels_like,
            'humidity' : humidity
        }

    else :  # 응답 불량
        
        weather_info = {
            'city' : city,
            'description': 'Not Found',
            'temperature': 'Not Found',
            'temperature_feels_like' : 'Not Found',
            'humidity' : 'Not Found'
        }

    return json.dumps(weather_info) # dict -> json

get_current_weather()        


'{"city": "Seoul", "description": "overcast clouds", "temperature": 32.76, "temperature_feels_like": 39.62, "humidity": 62}'

In [76]:
llm = init_chat_model('gpt-5.4-mini')

tools = [simple_calculator, get_current_weather]

agent = create_agent(
    llm, tools, 
    system_prompt='''
당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여,
사용자의 질문에 거짓없이 답변을 해주세요.
''')

response = agent.invoke({'messages' : '오늘 뭐 입어야 돼? 나 강원도에 살아.'})
pprint(response)
print("=" * 50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='오늘 뭐 입어야 돼? 나 강원도에 살아.', additional_kwargs={}, response_metadata={}, id='34e0528e-bbb0-4d03-92ac-5ba10cdd35c5'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 417, 'total_tokens': 441, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHOp5SwkAZ74mz4URsM90oZ43nVMA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0421e-ada2-7dc0-a202-12bfd8de6b98-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city': 'Gangwon-do', 'units': 'metric'}, 'id': '

In [ ]:
# 한국 기준 현재 날짜/시간을 반환하는 Tool

from datetime import datetime
from pytz import timezone

@tool
def get_current_datetime(format: str='%Y-%m-%d %H:%M:%S') -> str:
    """
    한국기준 현재시각정보를 반환하는 Tool
    Args:
        format: 날짜/시각 형식 지정
    Return:
        현재시각 문자열

    get_current_datetime() -> "2026-01-15 12:18:32"
    """

    kst = timezone('Asia/Seoul')  # 한국 시간대(KST) 설정
    # 현재 서울 시간을 받아, format 형식의 문자열로 반환
    return datetime.now(kst).strftime(format) 

get_current_datetime

StructuredTool(name='get_current_datetime', description='한국기준 현재시각정보를 반환하는 Tool\nArgs:\n    format: 날짜/시각 형식 지정\nReturn:\n    현재시각 문자열\n\nget_current_datetime() -> "2026-01-15 12:18:32"', args_schema=<class 'langchain_core.utils.pydantic.get_current_datetime'>, func=<function get_current_datetime at 0x00000196A4601E40>)

In [ ]:
@tool
def calculate_age(today_date: str, birth_date:str)->int:
    """
    오늘날짜, 생년월일을 입력받아 만나이를 계산하는 Tool
    Args:
        - today_date(str): 오늘 날짜 (yyyy-mm-dd형식)
        - birth_date(str): 생년월일 (yyyy-mm-dd형식)
    Return:
        - 계산된 만나이(int)
    """
    try:
        # 오늘 날짜 문자열 -> datetime 변환
        today = datetime.strptime(today_date, '%Y-%m-%d')
        # 생일 날짜 문자열 -> datetime 변환
        birthday = datetime.strptime(birth_date, '%Y-%m-%d')

        age = today.year - birthday.year # 기본 나이 계산

        # 생일이 아직 안지난 경우
        if (today.month, today.day) < (birthday.month, birthday.day):
            age -= 1 # 만나이는 -1
        return age 

    except ValueError:
        return "날짜 형식이 올바르지 않습니다. yyyy-mm-dd 형식으로 전달해 주세요."
    
calculate_age.invoke({
    "today_date": "2026-08-27",
    "birth_date": "1920-10-11"
})

105

In [ ]:
llm = init_chat_model('gpt-5.4-mini')

tools = load_tools(['wikipedia']) + [get_current_datetime, calculate_age]

agent = create_agent(
    llm, tools, 
    system_prompt='''
당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여,
사용자의 질문에 거짓없이 답변을 해주세요.
''')

response = agent.invoke(
    {'messages' : [('human','트럼프 대통령의 현재 나이는?')]},
    config = {'recursion_limit': 10} # ReAct 멀티턴 (툴 호출 반복) 최대횟수 제한
)
pprint(response)
print("=" * 50)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='트럼프 대통령의 현재 나이는?', additional_kwargs={}, response_metadata={}, id='90f4bf3d-6339-40ad-82bc-8de35d866f62'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 375, 'total_tokens': 428, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHPRDYmhjNwICz0XBqIxBAZVLOYeN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04242-be67-7d12-ac4e-3a5cc7d515b7-0', tool_calls=[{'name': 'get_current_datetime', 'args': {'format': '%Y-%m-%d'}, 'id': 'call_nvr38IrpbkY6mObrxog

## Memory
agent의 checkpointer속성에 메모리객체를 대화내역을 저장한다.
- 임시저장 InMemorySaver()
- 영구저장 SqliteSaver()

### InMemorySaver

In [96]:
from langgraph.checkpoint.memory import InMemorySaver # 메모리 기반 체크포인터 (대화 상태 저장)

llm = init_chat_model('gpt-5.4-mini')

tools = [TavilySearch()]

# 체크포인터를 전달하여 대화 상태 저장 가능하도록 에이전트 생성
agent = create_agent(llm, tools, checkpointer=InMemorySaver())

response = agent.invoke(
    input = {'messages': [('human', '안녕! 만나서 반갑다! 나는 cap이라고 해. 넌 누구니?')]},
    config = {'configurable' : {'thread_id':'100'}} # thread_id로 대화 식별
)

print(response['messages'][-1].content)


안녕, cap! 만나서 반가워. 나는 ChatGPT라고 해.  
궁금한 거 있으면 편하게 물어봐!


In [98]:
response = agent.invoke(
    input = {'messages': [('human', '어 그래 너 gpt 구나~ 내 이름은 김건우야')]},
    config = {'configurable': {'thread_id': '200'}} # thread_id 200번으로 새로운 대화 시작
)

print(response['messages'][-1].content)

반가워요, 김건우님 😄  
저는 GPT예요. 편하게 말씀해 주세요!


In [99]:
response = agent.invoke(
    input = {'messages': [('human', '어 그래 너 gpt 구나~ 내 이름이 뭐였지? 나 기억 상실증이야!!')]},
    config = {'configurable': {'thread_id': '200'}} # thread_id 200번으로 새로운 대화 시작
)

print(response['messages'][-1].content)

당연히 기억해요. 당신 이름은 **김건우**예요.


### sqliteSaver

In [ ]:
# Langgraph 상태 저장을 SQLite로 영속화하여 저장하는 체크포인터 패키지
%pip install -Uqqq langgraph-checkpoint-sqlite

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver # Sqlite 기반 체크포인트 (Saver)
from pprint import pprint

llm = init_chat_model('gpt-5.4-mini')
tools = [TavilySearch()]

# Sqlite DB 연결을 checkpoint.db 컨텍스트로 관리
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer.setup() # 테이블 생성 및 초기화

    # 에이전트가 상태 저장소로 checkpointer 활용
    agent = create_agent(llm, tools, checkpointer=checkpointer)

    response = agent.invoke(
        input = {'messages': [('human', 'Langchain 에 대해 설명해줘.')]},
        config = {'configurable': {'thread_id':'100'}} # thread_id 로 대화 식별
    )

    pprint(response)
    print("=" * 50)
    pprint(response['messages'][-1].content)
    print("=" * 100)

    response = agent.invoke(
        input = {'messages' : [('human', 'Langgraph 에 대해 설명해줘.')]},
        config={'configurable': {'thread_id':'100'}} # thread_id 로 대화 식별
    )
    
    pprint(response)
    print("=" * 50)
    pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='Langchain 에 대해 설명해줘.', additional_kwargs={}, response_metadata={}, id='735be110-7066-490b-b9c2-33724272c136'),
              AIMessage(content='LangChain은 **LLM(대규모 언어 모델)을 쉽게 연결하고 활용할 수 있게 해주는 프레임워크**입니다.  \n쉽게 말해, ChatGPT 같은 모델을 단순히 “질문-답변”으로 쓰는 것을 넘어서, **외부 데이터, API, 문서, DB, 검색, 에이전트 기능**과 연결해 더 복잡한 AI 앱을 만들도록 도와줍니다.\n\n## 핵심 개념\nLangChain이 잘하는 일은 보통 아래와 같습니다.\n\n1. **프롬프트 관리**\n   - LLM에 넣을 입력을 구조화하고 재사용하기 쉽게 만듭니다.\n   - 예: 질문, 역할, 문맥 등을 템플릿으로 구성\n\n2. **체인(Chain)**\n   - 여러 단계를 순서대로 연결합니다.\n   - 예: 문서 검색 → 요약 → 답변 생성\n\n3. **에이전트(Agent)**\n   - 모델이 상황에 따라 어떤 도구를 쓸지 스스로 판단하게 합니다.\n   - 예: 검색 도구, 계산기, DB 조회 도구 중 하나를 선택\n\n4. **RAG(Retrieval-Augmented Generation)**\n   - 외부 문서나 지식베이스에서 관련 정보를 찾아 답변에 반영합니다.\n   - 예: 사내 문서 기반 Q&A 챗봇\n\n5. **툴/외부 연동**\n   - API, 검색엔진, 데이터베이스, 파일 시스템 등과 연결할 수 있습니다.\n\n---\n\n## 왜 쓰나?\nLLM만 단독으로 쓰면 이런 한계가 있습니다.\n\n- 최신 정보에 약함\n- 회사 내부 문서를 모름\n- 계산이나 특정 작업 수행이 어려움\n- 복잡한 워크플로우를 처리하기 힘듦\n\nLangChain은 이런 문제를 해결하기 위해  \n**L

In [105]:
# 사용자가 재접속한 상황
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer.setup() # 테이블 생성 및 초기화 (기존에 존재하면 그대로 사용)

    # 에이전트가 상태 저장소로 checkpointer 활용
    agent = create_agent(llm, tools, checkpointer=checkpointer)

    response = agent.invoke(
        input = {'messages': [('human', '오케이 완전 이해했어! 그럼 니가 말해준 langchain, langgraph 를 세줄 요약해줘.')]},
        config = {'configurable': {'thread_id':'100'}} # thread_id 로 대화 식별
    )

    pprint(response)
    print("=" * 50)
    pprint(response['messages'][-1].content)
    print("=" * 100)

{'messages': [HumanMessage(content='Langchain 에 대해 설명해줘.', additional_kwargs={}, response_metadata={}, id='735be110-7066-490b-b9c2-33724272c136'),
              AIMessage(content='LangChain은 **LLM(대규모 언어 모델)을 쉽게 연결하고 활용할 수 있게 해주는 프레임워크**입니다.  \n쉽게 말해, ChatGPT 같은 모델을 단순히 “질문-답변”으로 쓰는 것을 넘어서, **외부 데이터, API, 문서, DB, 검색, 에이전트 기능**과 연결해 더 복잡한 AI 앱을 만들도록 도와줍니다.\n\n## 핵심 개념\nLangChain이 잘하는 일은 보통 아래와 같습니다.\n\n1. **프롬프트 관리**\n   - LLM에 넣을 입력을 구조화하고 재사용하기 쉽게 만듭니다.\n   - 예: 질문, 역할, 문맥 등을 템플릿으로 구성\n\n2. **체인(Chain)**\n   - 여러 단계를 순서대로 연결합니다.\n   - 예: 문서 검색 → 요약 → 답변 생성\n\n3. **에이전트(Agent)**\n   - 모델이 상황에 따라 어떤 도구를 쓸지 스스로 판단하게 합니다.\n   - 예: 검색 도구, 계산기, DB 조회 도구 중 하나를 선택\n\n4. **RAG(Retrieval-Augmented Generation)**\n   - 외부 문서나 지식베이스에서 관련 정보를 찾아 답변에 반영합니다.\n   - 예: 사내 문서 기반 Q&A 챗봇\n\n5. **툴/외부 연동**\n   - API, 검색엔진, 데이터베이스, 파일 시스템 등과 연결할 수 있습니다.\n\n---\n\n## 왜 쓰나?\nLLM만 단독으로 쓰면 이런 한계가 있습니다.\n\n- 최신 정보에 약함\n- 회사 내부 문서를 모름\n- 계산이나 특정 작업 수행이 어려움\n- 복잡한 워크플로우를 처리하기 힘듦\n\nLangChain은 이런 문제를 해결하기 위해  \n**L

In [107]:
# SQLite 체크포인터(DB)의 특정 thread_id의 대화 메시지 조회
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer_tuple = checkpointer.get_tuple({"configurable": {"thread_id" : "100"}})
    checkpointer_data = checkpointer_tuple.checkpoint
    messages = checkpointer_data['channel_values']['messages']

    for i, message in enumerate(messages, 1):
        msg_type = getattr(messages, 'type', messages.__class__.__name__)
        print(f"{i}: [{msg_type}] {message.content}")
        print()


1: [list] Langchain 에 대해 설명해줘.

2: [list] LangChain은 **LLM(대규모 언어 모델)을 쉽게 연결하고 활용할 수 있게 해주는 프레임워크**입니다.  
쉽게 말해, ChatGPT 같은 모델을 단순히 “질문-답변”으로 쓰는 것을 넘어서, **외부 데이터, API, 문서, DB, 검색, 에이전트 기능**과 연결해 더 복잡한 AI 앱을 만들도록 도와줍니다.

## 핵심 개념
LangChain이 잘하는 일은 보통 아래와 같습니다.

1. **프롬프트 관리**
   - LLM에 넣을 입력을 구조화하고 재사용하기 쉽게 만듭니다.
   - 예: 질문, 역할, 문맥 등을 템플릿으로 구성

2. **체인(Chain)**
   - 여러 단계를 순서대로 연결합니다.
   - 예: 문서 검색 → 요약 → 답변 생성

3. **에이전트(Agent)**
   - 모델이 상황에 따라 어떤 도구를 쓸지 스스로 판단하게 합니다.
   - 예: 검색 도구, 계산기, DB 조회 도구 중 하나를 선택

4. **RAG(Retrieval-Augmented Generation)**
   - 외부 문서나 지식베이스에서 관련 정보를 찾아 답변에 반영합니다.
   - 예: 사내 문서 기반 Q&A 챗봇

5. **툴/외부 연동**
   - API, 검색엔진, 데이터베이스, 파일 시스템 등과 연결할 수 있습니다.

---

## 왜 쓰나?
LLM만 단독으로 쓰면 이런 한계가 있습니다.

- 최신 정보에 약함
- 회사 내부 문서를 모름
- 계산이나 특정 작업 수행이 어려움
- 복잡한 워크플로우를 처리하기 힘듦

LangChain은 이런 문제를 해결하기 위해  
**LLM + 데이터 + 도구 + 로직**을 하나의 앱으로 묶어줍니다.

---

## 간단한 예시
예를 들어 “회사 규정에 대해 묻는 챗봇”을 만든다고 하면:

- 사용자의 질문을 받음
- 관련 규정 문서를 검색함
- 찾은 문서를 바탕으로 답변 생성
- 필요하면 추가 질문을 하거나 다른 도구를 호출

이런 흐름을 

1️⃣ 세션(메모리) 유지 방식
예: store = {}, ChatMessageHistory, InMemorySaver
- 특징
    - 서버 메모리에만 대화 상태를 저장
    - 서버 재시작/재배포 시 모두 사라짐
    - 구현이 가장 단순하고 빠름
- 사용 시기
    - 실습 / 데모 / PoC
    - 단일 서버, 짧은 대화
    - “지금 이 세션에서만 기억하면 되는” 경우
- 장단점
    - ✅ 속도 빠름, 구현 쉬움
    - ❌ 서버 내려가면 기억 소멸
    - ❌ 멀티 서버(스케일아웃) 불가능

2️⃣ SQLite 체크포인터
예: SqliteSaver, checkpoint.db
- 특징
    - 로컬 파일(DB)에 대화 상태 저장
    - 서버 재시작해도 대화 복원 가능
    - 설정/운영 부담이 거의 없음
- 사용 시기
    - 1대 서버 운영
    - “재접속 시 대화 이어가기”가 중요한 서비스
    - 내부 도구, 사내용 챗봇, 파일 기반 서비스
- 장단점
    - ✅ 재시작해도 대화 유지
    - ✅ 설정 간단 (파일 하나)
    - ❌ 동시접속/대량 트래픽에 취약
    - ❌ 운영·분석·확장성 한계

3️⃣ RDB (MySQL / PostgreSQL 등)
실무에서 가장 많이 쓰는 방식

- 특징
    - 대화 내역을 정규화된 테이블로 저장
    - 여러 서버가 공유 DB 사용 가능
    - 사용자/세션/대화/이력 분석까지 가능
- 사용 시기
    - 실서비스(운영 환경)
    - 로그인 사용자 기반 챗봇
    - 고객지원, 상담, 금융, 헬스케어, 교육 서비스
- 장단점
    - ✅ 서버 여러 대에서도 동일한 대화 유지
    - ✅ 로그/분석/감사/리포트 가능
    - ✅ 권한·보안·백업 체계화 가능
    - ❌ 설계/운영 비용 존재

- 요약하자면  
메모리 세션    : 빠르고 간단  
SQLite 같은 파일형 DB : 재접속 기억 + 운영 부담 최소  
RDB    같은 관계형 데이터베이스 : 확장성, 안정성, 분석, 운영  

- 서비스 구상 단계에서  
사용자별 히스토리 관리  
문제 발생 시 감사 로그  
대화 품질/모델 성능 분석  
요약/임베딩/재검색(RAG) 연계  
개인화 서비스(추천, 성향 파악)  
이걸 하려면 RDB 또는 그 이상(이벤트 로그, 데이터 웨어하우스) 가 필요